# Full-Sequence Sign Language Recognition with Leap Motion (LSTM, Palm Reference)

This notebook trains a **full-sequence unidirectional LSTM** on Leap Motion skeleton data (132 features, palm-reference normalization).

Pipeline stages:
1. Imports and reproducibility setup
2. Feature extraction from Leap CSV frames (132 raw features)
3. Segment loading from annotation TXT files
4. Segment extraction and quality filtering
5. Train/val/test split (dev users user1, user2, user5 + lockbox test user3)
6. Label encoding from training segments
7. Full-recording dataset with frame-level labels
8. LSTM training and frame-level evaluation
9. Online streaming inference + WER (causal, one frame at a time)


## 1) Imports and Setup


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import subprocess
import uuid
from collections import Counter, defaultdict, deque
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython import get_ipython
except Exception:
    get_ipython = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior can make training slower, but helps reproducibility.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2) Central Configuration, Project Paths, and Feature Definition


In [ ]:
# Central configuration

# Label used for unlabeled gaps between annotated signs.
BACKGROUND_LABEL = "background"

# Dataset split setup: train/validation come from the development users, test stays user-held-out.
DEV_USERS = ["user1", "user2", "user5"]
TEST_USER = "user3"
DEV_VAL_RATIO = 0.15
DEV_VAL_SEED = SEED + 202

# Sequence batching for full-recording LSTM training.
BATCH_SIZE = 4

# Core training setup for the full-sequence LSTM experiment.
MODEL_NAME = "lstm_fullseq"
NORMALIZATION_NAME = "palm_ref"
EPOCHS = 1
LEARNING_RATE = 3e-4

# LSTM model capacity and regularization.
FEAT_DIM = 128
HIDDEN_SIZE = 256
NUM_LSTM_LAYERS = 2
DROPOUT = 0.0

# Streaming / online inference setup used for WER evaluation.
STREAM_MODE = "lstm_online"
WER_EXAMPLE_PRINT_COUNT = 5

# Decoder timing assumptions based on Leap frame rate.
LEAP_FPS = 30
MIN_SIGN_MS = 500
MIN_SIGN_FRAMES = max(1, int(round((MIN_SIGN_MS / 1000.0) * LEAP_FPS)))

# Bag-based smoothing and gating used by the online decoder.
BAG_SIZE = 5
BAG_AGGREGATION = "mean"
CONFIDENCE_THRESHOLD = 0.35
SIGN_BG_MARGIN = 0.10


def find_project_root(start_path: Path | None = None) -> Path:
    """Find project root by searching upwards for a directory containing dataset/."""
    current = (start_path or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "dataset").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dataset/.")


PROJECT_ROOT = Path.cwd().parents[1]
DATASET_ROOT = PROJECT_ROOT / "dataset"
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")

HANDS = ["left", "right"]
FINGERS = ["thumb", "index", "middle", "ring", "pinky"]
BONES = ["metacarpal", "proximal", "intermediate", "distal"]
CARTESIAN_AXES = ["x", "y", "z"]
START_AXES = ["sx", "sy", "sz"]

FEATURE_KEYS = []

# Palm and wrist coordinates per hand.
for hand in HANDS:
    for part in ["palm", "wrist"]:
        for axis in CARTESIAN_AXES:
            FEATURE_KEYS.append(f"{hand}_{part}_{axis}")

# Finger bone start-joint coordinates per hand.
for hand in HANDS:
    for finger in FINGERS:
        for bone in BONES:
            for axis in START_AXES:
                FEATURE_KEYS.append(f"{hand}_{finger}_{bone}_{axis}")

assert len(FEATURE_KEYS) == 132, f"Expected 132 features, got {len(FEATURE_KEYS)}"
print(f"Feature count: {len(FEATURE_KEYS)}")

FEATURE_INDEX = {key: idx for idx, key in enumerate(FEATURE_KEYS)}
PALM_TRIPLETS: dict[str, tuple[int, int, int]] = {}
HAND_POSITION_TRIPLETS: dict[str, list[tuple[int, int, int]]] = {}

for hand in HANDS:
    PALM_TRIPLETS[hand] = tuple(FEATURE_INDEX[f"{hand}_palm_{axis}"] for axis in CARTESIAN_AXES)

    triplets: list[tuple[int, int, int]] = []
    triplets.append(tuple(FEATURE_INDEX[f"{hand}_wrist_{axis}"] for axis in CARTESIAN_AXES))
    for finger in FINGERS:
        for bone in BONES:
            triplets.append(tuple(FEATURE_INDEX[f"{hand}_{finger}_{bone}_{axis}"] for axis in START_AXES))
    HAND_POSITION_TRIPLETS[hand] = triplets

GCN_NODE_FEATURES = 3
GCN_NUM_NODES = len(FEATURE_KEYS) // GCN_NODE_FEATURES


def feature_window_to_gcn_nodes(window: np.ndarray) -> np.ndarray:
    """Convert (T, D) windows to (T, V, 3) for GCN input."""
    arr = np.asarray(window, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[1] != len(FEATURE_KEYS):
        raise ValueError(f"Expected window with shape (T, {len(FEATURE_KEYS)}), got {arr.shape}")
    return arr.reshape(arr.shape[0], GCN_NUM_NODES, GCN_NODE_FEATURES)


print("Palm-reference index map ready for GCN.")


## 3) Frame-Level Feature Extraction

The function below converts one Leap CSV row into a fixed (132,) numeric vector.
Missing columns or invalid values are safely converted to 0.0 to keep the pipeline robust.

In [ ]:
def _safe_float(value, default: float = 0.0) -> float:
    """Convert values to float safely; return default on invalid input."""
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def palm_reference_normalize_frame(frame: np.ndarray) -> np.ndarray:
    """Normalize one frame by subtracting each hand palm from that hand coordinates.

    Input shape: (132,)
    Output shape: (132,)
    """
    out = np.asarray(frame, dtype=np.float32).copy()
    if out.shape[0] != len(FEATURE_KEYS):
        raise ValueError(f"Expected frame with {len(FEATURE_KEYS)} features, got {out.shape[0]}")

    for hand in HANDS:
        px, py, pz = PALM_TRIPLETS[hand]
        palm = out[[px, py, pz]].copy()
        if not np.all(np.isfinite(palm)):
            palm = np.zeros((3,), dtype=np.float32)

        for ix, iy, iz in HAND_POSITION_TRIPLETS[hand]:
            out[ix] = out[ix] - palm[0]
            out[iy] = out[iy] - palm[1]
            out[iz] = out[iz] - palm[2]

        # Keep palm channels for stable shape and explicit origin anchoring.
        out[px] = 0.0
        out[py] = 0.0
        out[pz] = 0.0

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)


def palm_reference_normalize_sequence(sequence: np.ndarray) -> np.ndarray:
    """Apply palm-reference normalization frame-wise.

    Input shape: (T, D)
    Output shape: (T, D)
    """
    seq = np.asarray(sequence, dtype=np.float32)
    if seq.ndim != 2:
        raise ValueError(f"Expected sequence with shape (T, D), got {seq.shape}")
    if seq.shape[0] == 0:
        return seq.astype(np.float32, copy=False)

    normalized = np.empty_like(seq, dtype=np.float32)
    for i in range(seq.shape[0]):
        normalized[i] = palm_reference_normalize_frame(seq[i])
    return normalized


def extract_features_from_row(row: pd.Series) -> np.ndarray:
    """Convert one Leap CSV row to one 132-D raw feature frame (no normalization)."""
    values = np.asarray([_safe_float(row.get(key, 0.0)) for key in FEATURE_KEYS], dtype=np.float32)
    return np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)


# Quick structural self-check with a synthetic row.
test_row = pd.Series({key: 1.0 for key in FEATURE_KEYS[:5]})
test_features = extract_features_from_row(test_row)
print(f"Feature vector shape: {test_features.shape}")

## 4) Load Segmentation and Leap CSV Data

- load_segments(path) reads annotation TXT files (start_frame end_frame label).
- load_leap_csv(path) loads frame-level data and converts each frame to 132 raw features.

In [ ]:
def load_segments(path: Path) -> list[dict]:
    """Load segmentation file into a list of {start, end, label} dictionaries."""
    path = Path(path)
    if not path.exists():
        return []

    # The file format is whitespace-separated with no headers.
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["start", "end", "label"], engine="python")
    segments = []

    for _, row in df.iterrows():
        try:
            start = int(row["start"])
            end = int(row["end"])
            label = str(row["label"]).strip()
            if label:
                segments.append({"start": start, "end": end, "label": label})
        except Exception:
            # Skip malformed rows to avoid interrupting the full data scan.
            continue

    return segments


def load_leap_csv(path: Path, return_dataframe: bool = False):
    """Load Leap CSV and return feature matrix (num_frames, 132)."""
    path = Path(path)
    if not path.exists():
        empty = np.zeros((0, len(FEATURE_KEYS)), dtype=np.float32)
        return (empty, pd.DataFrame()) if return_dataframe else empty

    df = pd.read_csv(path)
    if df.empty:
        empty = np.zeros((0, len(FEATURE_KEYS)), dtype=np.float32)
        return (empty, df) if return_dataframe else empty

    feature_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {path.name}", leave=False):
        feature_rows.append(extract_features_from_row(row))

    features = np.vstack(feature_rows).astype(np.float32)
    return (features, df) if return_dataframe else features

## 5) Recording Discovery and Segment Extraction (Sign + Background)

Each user recording is matched by recording ID (for example `P1_S1_R1`) between:
- Leap CSV file
- Segmentation TXT file

For each recording, we build segment intervals for:
- Annotated sign glosses from segmentation files
- Background gloss from unannotated frame gaps between signs

This gives pseudo ground-truth intervals for both sign and background classes.

In [ ]:
RECORDING_ID_PATTERN = re.compile(r"(P\d+_S\d+_R\d+)", re.IGNORECASE)


def extract_recording_id(filename: str) -> str | None:
    """Extract canonical recording ID from a CSV or TXT filename."""
    match = RECORDING_ID_PATTERN.search(filename)
    return match.group(1) if match else None


def find_user_recordings(dataset_root: Path) -> dict[str, list[dict]]:
    """Find matching (CSV, segmentation) pairs for each user directory."""
    dataset_root = Path(dataset_root)
    user_map: dict[str, list[dict]] = defaultdict(list)

    for user_dir in sorted(dataset_root.glob("user*")):
        if not user_dir.is_dir():
            continue

        leap_dir = user_dir / "leap_data"
        seg_dir = user_dir / "segmentation"
        if not leap_dir.exists() or not seg_dir.exists():
            continue

        seg_map = {}
        for seg_path in seg_dir.glob("*.txt"):
            recording_id = extract_recording_id(seg_path.name)
            if recording_id is not None:
                seg_map[recording_id] = seg_path

        for csv_path in leap_dir.glob("*.csv"):
            recording_id = extract_recording_id(csv_path.name)
            if recording_id is None:
                continue

            seg_path = seg_map.get(recording_id)
            if seg_path is not None:
                user_map[user_dir.name].append({
                    "recording_id": recording_id,
                    "csv_path": csv_path,
                    "seg_path": seg_path,
                })

    return user_map


def build_intervals_with_background(segment_defs: list[dict], num_frames: int, background_label: str = BACKGROUND_LABEL) -> list[dict]:
    """Build labeled intervals including background gaps between annotated signs."""
    if num_frames <= 0:
        return []

    cleaned = []
    for seg in segment_defs:
        try:
            start = max(0, int(seg["start"]))
            end = min(num_frames - 1, int(seg["end"]))
            label = str(seg["label"]).strip()
            if end >= start and label:
                cleaned.append({"start": start, "end": end, "label": label, "is_background": False})
        except Exception:
            continue

    cleaned.sort(key=lambda x: (x["start"], x["end"]))
    intervals = []
    prev_end = -1

    for seg in cleaned:
        if seg["start"] > prev_end + 1:
            intervals.append({
                "start": prev_end + 1,
                "end": seg["start"] - 1,
                "label": background_label,
                "is_background": True,
            })

        intervals.append(seg)
        prev_end = max(prev_end, seg["end"])

    if prev_end < num_frames - 1:
        intervals.append({
            "start": prev_end + 1,
            "end": num_frames - 1,
            "label": background_label,
            "is_background": True,
        })

    return intervals


def extract_segments_for_recording(csv_path: Path, seg_path: Path) -> list[dict]:
    """Return per-interval records for one recording, including background gaps."""
    features, raw_df = load_leap_csv(csv_path, return_dataframe=True)
    segment_defs = load_segments(seg_path)

    if features.shape[0] == 0:
        return []

    interval_defs = build_intervals_with_background(segment_defs, num_frames=features.shape[0])
    if not interval_defs:
        return []

    confidence_cols = [col for col in ["left_confidence", "right_confidence"] if col in raw_df.columns]
    records = []

    for seg in interval_defs:
        start = max(0, int(seg["start"]))
        end = min(features.shape[0] - 1, int(seg["end"]))
        if end < start:
            continue

        segment_array = features[start:end + 1]
        if segment_array.size == 0:
            continue

        segment_confidence = None
        if confidence_cols:
            conf_slice = raw_df.loc[start:end, confidence_cols].to_numpy(dtype=np.float32)
            if conf_slice.size > 0:
                segment_confidence = np.nanmean(conf_slice, axis=1)

        records.append({
            "segment": segment_array.astype(np.float32),
            "label": seg["label"],
            "confidence": segment_confidence,
            "segment_span": (start, end),
            "recording_features": features,
            "is_background": bool(seg.get("is_background", False)),
        })

    return records


user_recordings = find_user_recordings(DATASET_ROOT)
if not user_recordings:
    raise RuntimeError("No matching Leap CSV and segmentation TXT pairs were found.")

print("Matched recordings per user:")
for user_name, recs in user_recordings.items():
    print(f"  {user_name}: {len(recs)} recordings")

segments_by_user: dict[str, list[dict]] = defaultdict(list)
for user_name, recordings in user_recordings.items():
    for rec in tqdm(recordings, desc=f"Extracting segments for {user_name}"):
        rec_segments = extract_segments_for_recording(rec["csv_path"], rec["seg_path"])
        for item in rec_segments:
            item["recording_id"] = rec["recording_id"]
        segments_by_user[user_name].extend(rec_segments)

all_segments_raw = [s for lst in segments_by_user.values() for s in lst]
segment_label_pairs = [(s["segment"], s["label"]) for s in all_segments_raw]

print(f"\nTotal raw segments (sign + background): {len(all_segments_raw)}")
print(f"Tuple storage format sample: (segment_array, label) -> {len(segment_label_pairs)} items")
print("Raw class distribution (segments):")
print(Counter([s["label"] for s in all_segments_raw]))


## 6) User-Based Split Setup + Segment Filtering

Before building train/validation/test sets, we define:
- **Development users**: `user1`, `user2`, and `user5` (pooled, then split into train/val)
- **Test user (lockbox)**: `user3`
- **Dev validation ratio**: `DEV_VAL_RATIO = 0.2` (recording-level split within dev users)

Segments are filtered by minimum average confidence before splitting.


In [ ]:


def dedupe_consecutive_labels(labels: list[str]) -> list[str]:
    """Collapse consecutive duplicate labels while preserving order."""
    if not labels:
        return []
    out = [labels[0]]
    for label in labels[1:]:
        if label != out[-1]:
            out.append(label)
    return out


def near_zero_frame_ratio(segment: np.ndarray, eps: float = 1e-6) -> float:
    """Compute fraction of frames whose L2 norm is almost zero."""
    if segment.size == 0:
        return 1.0
    frame_norm = np.linalg.norm(segment, axis=1)
    return float(np.mean(frame_norm < eps))


def build_recording_catalog(segments_by_user_map: dict[str, list[dict]], background_label: str = BACKGROUND_LABEL) -> list[dict]:
    """Aggregate interval-level records into recording-level continuous samples."""
    rec_map: dict[tuple[str, str], dict] = {}

    for user_name, items in segments_by_user_map.items():
        for item in items:
            recording_id = str(item.get("recording_id", "unknown"))
            key = (user_name, recording_id)
            if key not in rec_map:
                rec_map[key] = {
                    "user": user_name,
                    "recording_id": recording_id,
                    "V": item["recording_features"].astype(np.float32),
                    "intervals": [],
                }

            start, end = item["segment_span"]
            rec_map[key]["intervals"].append({
                "start": int(start),
                "end": int(end),
                "label": str(item["label"]),
                "is_background": bool(item.get("is_background", False)),
            })

    catalog = []
    for rec in rec_map.values():
        intervals = sorted(rec["intervals"], key=lambda x: (x["start"], x["end"]))
        segmentation_regions = [
            {
                "label": str(seg["label"]),
                "start_frame": int(seg["start"]),
                "end_frame": int(seg["end"]),
            }
            for seg in intervals
            if seg["label"] != background_label
        ]
        gt_labels = [seg["label"] for seg in segmentation_regions]
        gt_labels = dedupe_consecutive_labels(gt_labels)
        missing_ratio = near_zero_frame_ratio(rec["V"])
        catalog.append({
            "user": rec["user"],
            "recording_id": rec["recording_id"],
            "V": rec["V"],
            "ground_truth": gt_labels,
            "segmentation_regions": segmentation_regions,
            "missing_ratio": float(missing_ratio),
            "num_frames": int(rec["V"].shape[0]),
        })

    return catalog


def split_dev_recordings(
    catalog: list[dict],
    dev_users: list[str],
    val_ratio: float = DEV_VAL_RATIO,
    seed: int = DEV_VAL_SEED,
) -> tuple[set[tuple[str, str]], set[tuple[str, str]]]:
    """Split development recordings into train/val keys (recording-level, no window leakage)."""
    dev_recs = [
        rec for rec in catalog
        if rec["user"] in dev_users and len(rec.get("ground_truth", [])) > 0
    ]
    if len(dev_recs) == 0:
        raise RuntimeError("No development recordings with non-empty ground truth were found.")

    rng = random.Random(seed)
    rng.shuffle(dev_recs)

    n = len(dev_recs)
    n_val = max(1, int(round(n * val_ratio))) if n > 1 else 0
    if n_val >= n:
        n_val = max(1, n - 1)

    val_recs = dev_recs[:n_val]
    train_recs = dev_recs[n_val:]

    train_keys = {(rec["user"], rec["recording_id"]) for rec in train_recs}
    val_keys = {(rec["user"], rec["recording_id"]) for rec in val_recs}
    return train_keys, val_keys


def filter_segments(segments: list[dict], min_len: int = 10, max_zero_ratio: float = 0.40, min_confidence: float = 0.1):
    """Return filtered segments and a reason counter for removed segments."""
    kept = []
    removed_reasons = Counter()

    for item in segments:
        segment = item["segment"]
        conf = item.get("confidence", None)
        reasons = []

        if len(segment) < min_len:
            reasons.append("too_short")
        if near_zero_frame_ratio(segment) > max_zero_ratio:
            reasons.append("missing_data")
        if conf is not None and len(conf) > 0:
            avg_conf = float(np.nanmean(conf))
            if np.isnan(avg_conf) or avg_conf < min_confidence:
                reasons.append("low_confidence")

        if reasons:
            for reason in reasons:
                removed_reasons[reason] += 1
            continue

        kept.append(item)

    return kept, removed_reasons


available_users = sorted(segments_by_user.keys())
dataset_users = sorted([p.name for p in DATASET_ROOT.glob("user*") if p.is_dir()])
missing_dev_users = [u for u in DEV_USERS if u not in available_users]

if missing_dev_users:
    missing_detail = []
    for user_name in missing_dev_users:
        if user_name in dataset_users:
            missing_detail.append(
                f"{user_name} exists in dataset but has no matched CSV+TXT recording pairs"
            )
        else:
            missing_detail.append(f"{user_name} folder not found under dataset/")
    raise RuntimeError(
        "Missing required development users after recording discovery. "
        + "; ".join(missing_detail)
        + f". Available discovered users: {available_users}"
    )

if TEST_USER not in available_users:
    if TEST_USER in dataset_users:
        raise RuntimeError(
            f"Test user '{TEST_USER}' exists in dataset but has no matched CSV+TXT recording pairs. "
            f"Discovered users with matched pairs: {available_users}"
        )
    raise RuntimeError(
        f"Test user '{TEST_USER}' folder is missing under dataset/. "
        f"Available discovered users: {available_users}"
    )

recording_catalog = build_recording_catalog(segments_by_user)
if len(recording_catalog) == 0:
    raise RuntimeError("No recording-level continuous samples were found.")

dev_train_recording_keys, dev_val_recording_keys = split_dev_recordings(
    catalog=recording_catalog,
    dev_users=DEV_USERS,
    val_ratio=DEV_VAL_RATIO,
    seed=DEV_VAL_SEED,
)

# Raw segment pools before filtering.
dev_segments_pool: list[dict] = []
test_segments_pool: list[dict] = []

for user_name, user_segments in segments_by_user.items():
    for item in user_segments:
        item_with_user = dict(item)
        item_with_user["user"] = user_name
        if user_name in DEV_USERS:
            dev_segments_pool.append(item_with_user)
        elif user_name == TEST_USER:
            test_segments_pool.append(item_with_user)

dev_segments_filtered, dev_removed = filter_segments(dev_segments_pool)
test_segments_filtered, test_removed = filter_segments(test_segments_pool)

# Route dev segments by recording key; test segments all go to test.
filtered_train_segments: list[dict] = []
filtered_val_segments: list[dict] = []

for item in dev_segments_filtered:
    key = (item.get("user", "unknown"), str(item.get("recording_id", "unknown")))
    if key in dev_train_recording_keys:
        filtered_train_segments.append(item)
    elif key in dev_val_recording_keys:
        filtered_val_segments.append(item)

filtered_test_segments = list(test_segments_filtered)

test_raw_count = len(test_segments_pool)

print("========== TRAIN / VAL / TEST SPLIT (RECORDING-LEVEL) ==========")
print(f"Development users       : {DEV_USERS}")
print(f"Test user (lockbox)     : {TEST_USER}")
print(f"DEV_VAL_RATIO           : {DEV_VAL_RATIO}")
print(f"DEV_VAL_SEED            : {DEV_VAL_SEED}")
print(f"Total recordings in catalog: {len(recording_catalog)}")
print(f"Dev train recordings    : {len(dev_train_recording_keys)}")
print(f"Dev val recordings      : {len(dev_val_recording_keys)}")
print(f"Test recordings (user3) : {len([r for r in recording_catalog if r['user'] == TEST_USER])}")
print(f"Dev removed by filter   : {dict(dev_removed)}")
print(f"Test removed by filter  : {dict(test_removed)}")
print(f"Train segments (filtered): {len(filtered_train_segments)}")
print(f"Val segments (filtered)  : {len(filtered_val_segments)}")
print(f"Test segments (filtered) : {len(filtered_test_segments)}")

if len(filtered_train_segments) == 0:
    raise RuntimeError("Training segment pool is empty. Check DEV_VAL_RATIO or development data.")
if len(filtered_val_segments) == 0:
    raise RuntimeError("Validation segment pool is empty. Check DEV_VAL_RATIO or development data.")
if len(filtered_test_segments) == 0:
    raise RuntimeError("Test segment pool is empty. Check TEST_USER and data availability.")

print("Sample dev val recording IDs:", sorted({r for _, r in dev_val_recording_keys})[:5])
print("Sample test recording IDs:", sorted({r["recording_id"] for r in recording_catalog if r["user"] == TEST_USER})[:5])

# WER catalogs (continuous recordings with gloss GT).
test_wer_catalog = [
    rec for rec in recording_catalog
    if rec["user"] == TEST_USER and len(rec.get("ground_truth", [])) > 0
]
dev_val_wer_catalog = [
    rec for rec in recording_catalog
    if (rec["user"], rec["recording_id"]) in dev_val_recording_keys and len(rec.get("ground_truth", [])) > 0
]
dev_train_wer_catalog = [
    rec for rec in recording_catalog
    if (rec["user"], rec["recording_id"]) in dev_train_recording_keys and len(rec.get("ground_truth", [])) > 0
]

print(f"WER catalog sizes -> train(dev): {len(dev_train_wer_catalog)}, val(dev): {len(dev_val_wer_catalog)}, test: {len(test_wer_catalog)}")

# Per-user filtered segment maps for reporting.
filtered_train_segments_by_user: dict[str, list[dict]] = defaultdict(list)
filtered_val_segments_by_user: dict[str, list[dict]] = defaultdict(list)
filtered_test_segments_by_user: dict[str, list[dict]] = defaultdict(list)

for item in filtered_train_segments:
    filtered_train_segments_by_user[str(item.get("user", "unknown"))].append(item)
for item in filtered_val_segments:
    filtered_val_segments_by_user[str(item.get("user", "unknown"))].append(item)
for item in filtered_test_segments:
    filtered_test_segments_by_user[str(item.get("user", "unknown"))].append(item)

dev_segments_all: dict[str, list[dict]] = defaultdict(list)
for item in dev_segments_filtered:
    dev_segments_all[str(item.get("user", "unknown"))].append(item)

print("Filtered class distribution (train segments):")
print(Counter([s["label"] for s in filtered_train_segments]))
print("Filtered class distribution (val segments):")
print(Counter([s["label"] for s in filtered_val_segments]))
print("Filtered class distribution (test segments):")
print(Counter([s["label"] for s in filtered_test_segments]))


## 7) Final Split: Dev Train / Dev Val / Test (`user3`)

Final segment sets:
- **Train**: dev users (`user1`, `user2`, `user5`) — training portion after recording-level split
- **Val**: dev users — validation portion
- **Test**: `user3` only (lockbox)

WER evaluation catalogs are built from full recordings for streaming inference.


In [ ]:
# Build final train/val/test segment sets.
train_segments = list(filtered_train_segments)
val_segments = list(filtered_val_segments)
test_segments = list(filtered_test_segments)

if len(train_segments) == 0:
    raise RuntimeError("Training split is empty. Check DEV_VAL_RATIO and filtering thresholds.")
if len(val_segments) == 0:
    raise RuntimeError("Validation split is empty. Check DEV_VAL_RATIO and filtering thresholds.")
if len(test_segments) == 0:
    raise RuntimeError("Test split is empty. Check TEST_USER and filtering thresholds.")

train_segments_by_user = {"all_users": train_segments}
val_segments_by_user = {"all_users": val_segments}
test_segments_by_user = {"all_users": test_segments}

print("========== FINAL SPLIT (SEGMENTS) ==========")
print(f"Development train users: {DEV_USERS}")
for user_name in DEV_USERS:
    print(f"  train {user_name}: {len(filtered_train_segments_by_user.get(user_name, []))} segments")
print(f"Development val users  : {DEV_USERS}")
for user_name in DEV_USERS:
    print(f"  val   {user_name}: {len(filtered_val_segments_by_user.get(user_name, []))} segments")
print(f"Test user              : {TEST_USER}")
print(f"  test  {TEST_USER}: {len(filtered_test_segments_by_user.get(TEST_USER, []))} segments")

print(f"Train segments total : {len(train_segments)}")
print(f"Val segments total   : {len(val_segments)}")
print(f"Test segments total  : {len(test_segments)}")

print("Train class distribution (segments):")
print(Counter([s["label"] for s in train_segments]))
print("Val class distribution (segments):")
print(Counter([s["label"] for s in val_segments]))
print("Test class distribution (segments):")
print(Counter([s["label"] for s in test_segments]))


## 8) Label Encoding from Segments


In [ ]:

sign_labels = sorted({
    item["label"]
    for item in train_segments
    if not item.get("is_background", False)
})
all_labels = [BACKGROUND_LABEL] + sign_labels

label_encoder = LabelEncoder()
label_encoder.fit(all_labels)

label_to_id = {label: int(idx) for idx, label in enumerate(label_encoder.classes_)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
BACKGROUND_ID = int(label_to_id[BACKGROUND_LABEL])
NUM_CLASSES = len(label_to_id)

print("Label mapping (string -> ID):")
for label, idx in sorted(label_to_id.items(), key=lambda x: x[1]):
    print(f"  {label!r} -> {idx}")

print(f"\nTotal classes: {NUM_CLASSES}")
print(f"Background class ID: {BACKGROUND_ID}")

assert BACKGROUND_ID == 20, f"Expected BACKGROUND_ID==0, got {BACKGROUND_ID}"
assert NUM_CLASSES == len(sign_labels) + 1

missing = {
    item["label"]
    for item in train_segments
    if not item.get("is_background", False) and item["label"] not in label_to_id
}
assert not missing, f"Missing labels in label_to_id: {missing}"


## 9) Full-Sequence Dataset


In [ ]:


class FullSequenceDataset(Dataset):
    """One complete recording per item with frame-level integer labels."""

    def __init__(
        self,
        segments: list[dict],
        label_to_id: dict[str, int],
        normalize_fn=None,
        background_id: int = BACKGROUND_ID,
    ):
        self.label_to_id = label_to_id
        self.normalize_fn = normalize_fn
        self.background_id = background_id
        self.samples: list[dict] = []

        by_recording: dict[str, list[dict]] = defaultdict(list)
        for item in segments:
            rec_id = str(item.get("recording_id", "unknown"))
            by_recording[rec_id].append(item)

        for rec_id, items in sorted(by_recording.items()):
            video = items[0].get("recording_features", None)
            if video is None:
                continue

            video = np.asarray(video, dtype=np.float32)
            t_len = int(video.shape[0])
            labels = np.full(t_len, background_id, dtype=np.int64)

            sign_items = [
                it for it in items
                if not it.get("is_background", False)
            ]
            sign_items.sort(key=lambda it: it["segment_span"][0])

            for seg in sign_items:
                start, end = seg["segment_span"]
                sign_id = label_to_id[seg["label"]]
                labels[start : end + 1] = sign_id

            ground_truth = [seg["label"] for seg in sign_items]

            self.samples.append({
                "recording_id": rec_id,
                "user": items[0].get("user", "unknown"),
                "video": video,
                "labels": labels,
                "length": t_len,
                "ground_truth": ground_truth,
            })

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        video = sample["video"]
        if self.normalize_fn is not None:
            video = self.normalize_fn(video)
        video_t = torch.tensor(video, dtype=torch.float32)
        labels_t = torch.tensor(sample["labels"], dtype=torch.long)
        length_t = torch.tensor(sample["length"], dtype=torch.long)
        return video_t, labels_t, length_t


def collate_full_sequences(batch):
    videos, labels, lengths = zip(*batch)
    lengths = torch.stack(lengths)
    max_len = int(lengths.max().item())

    bsz = len(batch)
    feat_dim = videos[0].shape[1]
    padded_videos = torch.zeros(bsz, max_len, feat_dim, dtype=torch.float32)
    padded_labels = torch.full((bsz, max_len), -1, dtype=torch.long)

    for i, (video, label, length) in enumerate(zip(videos, labels, lengths)):
        seq_len = int(length.item())
        padded_videos[i, :seq_len] = video[:seq_len]
        padded_labels[i, :seq_len] = label[:seq_len]

    return padded_videos, padded_labels, lengths


def _print_dataset_stats(name: str, ds: FullSequenceDataset) -> None:
    if len(ds) == 0:
        print(f"{name}: 0 recordings loaded")
        if name.endswith("train") and len(train_segments) > 0:
            print("  keys:", list(train_segments[0].keys()))
        return

    lengths = [s["length"] for s in ds.samples]
    print(f"{name}: {len(ds)} recordings")
    print(f"  frame length min/max/mean: {min(lengths)}/{max(lengths)}/{np.mean(lengths):.1f}")

    counts = Counter()
    for s in ds.samples:
        counts.update(s["labels"].tolist())

    print(f"  frame-level class distribution ({len(counts)} classes):")
    for cid in sorted(counts):
        lbl = id_to_label.get(cid, str(cid))
        print(f"    ID {cid:2d} ({lbl:12s}): {counts[cid]:6d} frames")


lstm_train_ds = FullSequenceDataset(
    train_segments, label_to_id, normalize_fn=palm_reference_normalize_sequence
)
lstm_val_ds = FullSequenceDataset(
    val_segments, label_to_id, normalize_fn=palm_reference_normalize_sequence
)
lstm_test_ds = FullSequenceDataset(
    test_segments, label_to_id, normalize_fn=palm_reference_normalize_sequence
)

_print_dataset_stats("lstm_train_ds", lstm_train_ds)
_print_dataset_stats("lstm_val_ds", lstm_val_ds)
_print_dataset_stats("lstm_test_ds", lstm_test_ds)


## 10) DataLoaders


In [ ]:

lstm_train_loader = DataLoader(
    lstm_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_full_sequences,
)
lstm_val_loader = DataLoader(
    lstm_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_full_sequences,
)
lstm_test_loader = DataLoader(
    lstm_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_full_sequences,
)

print(f"Train batches: {len(lstm_train_loader)}")
print(f"Val batches  : {len(lstm_val_loader)}")
print(f"Test batches : {len(lstm_test_loader)}")


## 11) Class Weights and Loss Function


In [ ]:
frame_counts = Counter()
for sample in lstm_train_ds.samples:
    frame_counts.update(sample["labels"].tolist())

epsilon = 1e-6
raw_weights = np.array(
    [1.0 / (frame_counts.get(c, 0) + epsilon) for c in range(NUM_CLASSES)],
    dtype=np.float32,
)
class_weights = raw_weights * (NUM_CLASSES / raw_weights.sum())
class_weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weight_tensor, ignore_index=-1)

print("Raw frame counts per class:")
for c in range(NUM_CLASSES):
    print(f"  ID {c:2d} ({id_to_label[c]:12s}): {frame_counts.get(c, 0):6d}")

print("\nNormalized class weights:")
for c in range(NUM_CLASSES):
    print(f"  ID {c:2d} ({id_to_label[c]:12s}): {class_weights[c]:.6f}")


## 12) FullSequenceLSTM Model


In [ ]:
NORMALIZE_FN = palm_reference_normalize_sequence


In [ ]:

try:
    ipynb_name = get_ipython().user_ns.get("__vsc_ipynb_file__", "")
    if not ipynb_name:
        ipynb_name = get_ipython().user_ns.get("__file__", "")
    notebook_name = Path(ipynb_name).name.replace(".ipynb", "") if ipynb_name else "prototype_sequence_lstm_palm_ref_users3"
except Exception:
    notebook_name = "prototype_sequence_lstm_palm_ref_users3"



class FullSequenceLSTM(nn.Module):
    """Frame encoder -> unidirectional LSTM -> per-frame classifier."""

    def __init__(
        self,
        input_dim: int = 132,
        feat_dim: int = FEAT_DIM,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LSTM_LAYERS,
        num_classes: int = NUM_CLASSES,
        dropout: float = DROPOUT,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.feat_dim = feat_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_classes = num_classes

        self.frame_enc = nn.Sequential(
            nn.Linear(input_dim, feat_dim),
            nn.LayerNorm(feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.LayerNorm(hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes),
        )

    def _encode_frames(self, x: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        enc = self.frame_enc(x.reshape(b * t, d))
        return enc.reshape(b, t, self.feat_dim)

    def forward(self, sequences: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        enc = self._encode_frames(sequences)
        packed = nn.utils.rnn.pack_padded_sequence(
            enc, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        b, t, h = out.shape
        logits = self.classifier(out.reshape(b * t, h))
        return logits.reshape(b, t, self.num_classes)
    
    def step(self, frame: torch.Tensor, hidden=None):
        """Online inference: one normalized frame (1, 132) -> logits (1, C)."""
        # Ensure shape is (batch=1, seq_len=1, features)
        if frame.ndim == 1:
            # (132,) → (1, 1, 132)
            frame = frame.unsqueeze(0).unsqueeze(0)
        elif frame.ndim == 2:
            # (1, 132) → (1, 1, 132)
            frame = frame.unsqueeze(1)
        # if already (1, 1, 132), pass through

        enc = self.frame_enc(frame)              # (1, 1, 128)
        out, hidden = self.lstm(enc, hidden)     # out: (1, 1, 256)
        logits = self.classifier(out[:, -1, :]) # (1, 256) → (1, NUM_CLASSES)
        return logits, hidden


model = FullSequenceLSTM().to(DEVICE)

# Shape sanity checks
with torch.no_grad():
    dummy = torch.randn(2, 50, 132, device=DEVICE)
    dummy_lengths = torch.tensor([50, 40], device=DEVICE)
    out = model(dummy, dummy_lengths)
    assert out.shape == (2, 50, NUM_CLASSES), f"Bad forward shape: {out.shape}"

    frame = torch.randn(1, 132, device=DEVICE)
    logits, hidden = model.step(frame, None)
    assert logits.shape == (1, NUM_CLASSES), f"Bad step shape: {logits.shape}"

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME} | normalization: {NORMALIZATION_NAME}")
print(f"Trainable parameters: {trainable:,}")
print("Shape checks passed.")


## 13) Training


In [ ]:
def evaluate_lstm(model_obj: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model_obj.eval()
    total_loss = 0.0
    total_correct = 0
    total_frames = 0

    with torch.no_grad():
        for sequences, labels, lengths in loader:
            sequences = sequences.to(DEVICE)
            labels = labels.to(DEVICE)
            lengths = lengths.to(DEVICE)

            logits = model_obj(sequences, lengths)
            flat_logits = logits.reshape(-1, NUM_CLASSES)
            flat_labels = labels.reshape(-1)
            loss = criterion(flat_logits, flat_labels)
            total_loss += loss.item()

            mask = flat_labels != -1
            if mask.any():
                preds = torch.argmax(flat_logits, dim=1)
                total_correct += (preds[mask] == flat_labels[mask]).sum().item()
                total_frames += mask.sum().item()

    if total_frames == 0:
        return 0.0, 0.0

    acc = total_correct / total_frames
    avg_loss = total_loss / max(1, len(loader))
    return float(acc), float(avg_loss)


def train_lstm_model(
    model_obj: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LEARNING_RATE,
    use_grad_clip: bool = True,
    grad_clip_norm: float = 1.0,
    use_scheduler: bool = True,
    scheduler_patience: int = 5,
    scheduler_factor: float = 0.5,
    early_stopping_patience: int = 15,
) -> dict:
    if len(train_loader) == 0:
        raise RuntimeError("Training loader is empty.")

    optimizer = torch.optim.Adam(model_obj.parameters(), lr=lr)
    scheduler = None
    if use_scheduler:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=scheduler_factor, patience=scheduler_patience
        )

    best_val_acc = -1.0
    best_model_state = None
    epochs_without_improvement = 0

    history_train_loss = []
    history_train_acc = []
    history_train_batch_acc = []
    history_val_acc = []
    epoch_history = []

    print(f"[{MODEL_NAME}] using normalization: {NORMALIZATION_NAME}")
    print(
        f"Stabilization: grad_clip={use_grad_clip} (norm={grad_clip_norm}), "
        f"scheduler={use_scheduler}, early_stopping_patience={early_stopping_patience}"
    )

    for epoch in range(1, epochs + 1):
        model_obj.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for sequences, labels, lengths in tqdm(
            train_loader, desc=f"{MODEL_NAME} | Epoch {epoch}/{epochs}", leave=False
        ):
            sequences = sequences.to(DEVICE)
            labels = labels.to(DEVICE)
            lengths = lengths.to(DEVICE)

            optimizer.zero_grad()
            logits = model_obj(sequences, lengths)
            flat_logits = logits.reshape(-1, NUM_CLASSES)
            flat_labels = labels.reshape(-1)
            loss = criterion(flat_logits, flat_labels)
            loss.backward()

            if use_grad_clip:
                torch.nn.utils.clip_grad_norm_(model_obj.parameters(), max_norm=grad_clip_norm)

            optimizer.step()

            running_loss += loss.item()
            mask = flat_labels != -1
            if mask.any():
                batch_preds = torch.argmax(flat_logits, dim=1)
                running_correct += (batch_preds[mask] == flat_labels[mask]).sum().item()
                running_total += mask.sum().item()

        epoch_loss = running_loss / max(1, len(train_loader))
        history_train_loss.append(epoch_loss)

        epoch_train_batch_acc = running_correct / max(1, running_total)
        history_train_batch_acc.append(epoch_train_batch_acc)
        history_train_acc.append(float("nan"))

        val_acc, _ = evaluate_lstm(model_obj, val_loader)
        history_val_acc.append(val_acc)

        if scheduler is not None:
            scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.detach().clone() for k, v in model_obj.state_dict().items()}
            epochs_without_improvement = 0
            print(f"  -> [{MODEL_NAME}] new best in memory (val_acc={val_acc:.4f})")
        else:
            epochs_without_improvement += 1

        epoch_history.append({
            "model": MODEL_NAME,
            "normalization": NORMALIZATION_NAME,
            "epoch": epoch,
            "train_loss": float(epoch_loss),
            "train_batch_acc": float(epoch_train_batch_acc),
            "val_acc": float(val_acc),
        })

        print(
            f"{MODEL_NAME} | Epoch {epoch:02d} | Train Loss: {epoch_loss:.4f} | "
            f"Train Batch Acc: {epoch_train_batch_acc:.4f} | Val Acc: {val_acc:.4f}"
        )

        if early_stopping_patience is not None and epochs_without_improvement >= early_stopping_patience:
            print(
                f"Early stopping triggered after {epoch} epochs "
                f"(no improvement for {early_stopping_patience} epochs)."
            )
            break

    history_df_local = pd.DataFrame(epoch_history)
    print(f"\n{MODEL_NAME} per-epoch summary:")
    display(history_df_local.tail(min(10, len(history_df_local))))

    if best_model_state is not None:
        model_obj.load_state_dict(best_model_state)
        print(f"[{MODEL_NAME}] best model restored from memory.")

    return {
        "model_name": MODEL_NAME,
        "normalization": NORMALIZATION_NAME,
        "best_val_acc": float(best_val_acc),
        "best_model_state": best_model_state,
        "history_train_loss": history_train_loss,
        "history_train_acc": history_train_acc,
        "history_train_batch_acc": history_train_batch_acc,
        "history_val_acc": history_val_acc,
        "epoch_history": epoch_history,
    }


print("\n" + "=" * 70)
print(f"Training model: {MODEL_NAME}")
print(f"Normalization : {NORMALIZATION_NAME}")
print("=" * 70)

train_result = train_lstm_model(
    model_obj=model,
    train_loader=lstm_train_loader,
    val_loader=lstm_val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    use_grad_clip=True,
    grad_clip_norm=1.0,
    early_stopping_patience=15,
)

best_model_state = train_result["best_model_state"]
if best_model_state is not None:
    model.load_state_dict(best_model_state)

training_comparison_df = pd.DataFrame([
    {
        "model_name": MODEL_NAME,
        "normalization": train_result["normalization"],
        "best_val_acc": train_result["best_val_acc"],
        "final_val_acc": train_result["history_val_acc"][-1] if train_result["history_val_acc"] else 0.0,
    }
])
print("\n========== TRAINING SUMMARY ==========")
display(training_comparison_df)

history_train_loss = train_result["history_train_loss"]
history_train_acc = train_result["history_train_acc"]
history_train_batch_acc = train_result["history_train_batch_acc"]
history_val_acc = train_result["history_val_acc"]
epoch_history = train_result["epoch_history"]
history_df = pd.DataFrame(epoch_history)


## 14) Training Curves


In [ ]:
epochs_axis = np.arange(1, len(history_train_loss) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_axis, history_train_loss, marker="o")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
if "history_train_batch_acc" in globals() and len(history_train_batch_acc) == len(epochs_axis):
    plt.plot(epochs_axis, history_train_batch_acc, marker="x", linestyle="--", label="Train (batch-sampled)")
plt.plot(epochs_axis, history_val_acc, marker="o", label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 15) Frame-Level Evaluation


In [ ]:
def evaluate_lstm_full(
    model_obj: nn.Module,
    loader: DataLoader,
    split_name: str,
) -> tuple[float, np.ndarray, np.ndarray]:
    model_obj.eval()
    all_preds: list[int] = []
    all_targets: list[int] = []

    with torch.no_grad():
        for sequences, labels, lengths in loader:
            sequences = sequences.to(DEVICE)
            labels = labels.to(DEVICE)
            lengths = lengths.to(DEVICE)

            logits = model_obj(sequences, lengths)
            preds = torch.argmax(logits, dim=2)

            for i in range(sequences.size(0)):
                seq_len = int(lengths[i].item())
                all_preds.extend(preds[i, :seq_len].cpu().numpy().tolist())
                all_targets.extend(labels[i, :seq_len].cpu().numpy().tolist())

    if len(all_targets) == 0:
        return 0.0, np.array([], dtype=np.int64), np.array([], dtype=np.int64)

    acc = accuracy_score(all_targets, all_preds)
    return float(acc), np.asarray(all_preds, dtype=np.int64), np.asarray(all_targets, dtype=np.int64)


for split_name, loader in [
    ("Train", lstm_train_loader),
    ("Val", lstm_val_loader),
    ("Test", lstm_test_loader),
]:
    acc, preds, targets = evaluate_lstm_full(model, loader, split_name)
    print(f"\n========== {split_name.upper()} FRAME-LEVEL REPORT ==========")
    print(f"Accuracy: {acc:.4f}")
    target_names = [id_to_label[i] for i in range(NUM_CLASSES)]
    print(classification_report(targets, preds, target_names=target_names, zero_division=0))

summary_rows = []
for split_name, loader in [("Train", lstm_train_loader), ("Val", lstm_val_loader), ("Test", lstm_test_loader)]:
    acc, _, _ = evaluate_lstm_full(model, loader, split_name)
    summary_rows.append({"split": split_name, "frame_accuracy": acc})

frame_eval_summary_df = pd.DataFrame(summary_rows)
print("\n========== FRAME ACCURACY SUMMARY ==========")
display(frame_eval_summary_df)


## 16) Memorization Diagnostic


In [ ]:
# def remove_consecutive_duplicates(labels: list[str]) -> list[str]:
#     if not labels:
#         return []
#     out = [labels[0]]
#     for lbl in labels[1:]:
#         if lbl != out[-1]:
#             out.append(lbl)
#     return out


# def remove_background(labels: list[str], background_label: str = BACKGROUND_LABEL) -> list[str]:
#     return [lbl for lbl in labels if lbl != background_label]


# def _decode_frame_predictions(frame_ids: np.ndarray) -> list[str]:
#     labels = [id_to_label[int(i)] for i in frame_ids]
#     collapsed = remove_consecutive_duplicates(labels)
#     return remove_background(collapsed, BACKGROUND_LABEL)


# def _predict_recording_frames(model_obj: nn.Module, video: np.ndarray) -> np.ndarray:
#     model_obj.eval()
#     hidden = None
#     preds = []
#     with torch.no_grad():
#         for t in range(video.shape[0]):
#             frame = palm_reference_normalize_frame(video[t : t + 1])
#             frame_t = torch.tensor(frame, dtype=torch.float32, device=DEVICE)
#             logits, hidden = model_obj.step(frame_t, hidden)
#             preds.append(int(torch.argmax(logits, dim=1).item()))
#     return np.asarray(preds, dtype=np.int64)


# print("========== MEMORIZATION DIAGNOSTIC ==========")
# sample_indices = list(range(min(3, len(lstm_train_ds))))

# for idx in sample_indices:
#     sample = lstm_train_ds.samples[idx]
#     video = sample["video"]
#     gt = sample["ground_truth"]

#     orig_preds = _predict_recording_frames(model, video)
#     shuffled = video.copy()
#     rng = np.random.RandomState(SEED + idx)
#     rng.shuffle(shuffled)
#     shuf_preds = _predict_recording_frames(model, shuffled)

#     orig_seq = _decode_frame_predictions(orig_preds)
#     shuf_seq = _decode_frame_predictions(shuf_preds)

#     print(f"\nRecording: {sample['recording_id']} ({sample['user']})")
#     print(f"  Ground truth       : {gt}")
#     print(f"  Original prediction: {orig_seq}")
#     print(f"  Shuffled prediction: {shuf_seq}")


## 17) Online Streaming WER


In [ ]:
# ---------------------------------------------------------------------------
# Bag aggregator — unchanged from THCT-Net version
# ---------------------------------------------------------------------------

class _BagAggregator:
    """
    Causal sliding bag over raw logits.

    Why logits and not probs:
        Averaging in logit space is equivalent to a product-of-experts,
        which is sharper and more discriminative than averaging softmax probs.
        Converting to probs happens once after aggregation.

    Modes
    -----
    mean      : arithmetic mean of per-window probs after softmax
    max       : element-wise max of per-window probs
    attention : recency-weighted mean, most recent window weighted highest
    """

    def __init__(self, bag_size: int, aggregation: str, num_classes: int):
        self.bag_size    = max(1, int(bag_size))
        self.aggregation = aggregation
        self.num_classes = num_classes
        self._buffer     = deque(maxlen=self.bag_size)

    def update(self, logits: np.ndarray) -> np.ndarray | None:
        """
        Push one logit vector and return aggregated probs.

        Returns None until bag is full (first bag_size frames are skipped).
        """
        self._buffer.append(logits.copy())

        if len(self._buffer) < self.bag_size:
            return None

        bag         = np.stack(self._buffer, axis=0)           # (bag_size, C)
        bag_shifted = bag - bag.max(axis=-1, keepdims=True)
        exp_bag     = np.exp(bag_shifted)
        probs       = exp_bag / exp_bag.sum(axis=-1, keepdims=True)  # (bag_size, C)

        if self.aggregation == "mean":
            return probs.mean(axis=0)

        if self.aggregation == "max":
            return probs.max(axis=0)

        if self.aggregation == "attention":
            weights  = np.linspace(0.5, 1.0, len(self._buffer))
            weights /= weights.sum()
            return (probs * weights[:, np.newaxis]).sum(axis=0)

        raise ValueError(f"Unknown aggregation mode: {self.aggregation}")

    def reset(self):
        self._buffer.clear()


# ---------------------------------------------------------------------------
# Simplified streaming decoder
# ---------------------------------------------------------------------------

class SimplifiedBagDecoder:
    """
    Causal streaming decoder using bag-aggregated logits.

    States
    ------
    SEEKING : waiting for a sign to begin
    IN_SIGN : inside an active sign region, accumulating votes

    Emission
    --------
    Fires at the TRAILING edge when the bag transitions to background.
    Emits the majority label observed across the entire region.
    Discards regions shorter than min_sign_frames (noise / glitches).

    Storage additions vs original
    -----------------------------
    pre_bag_logits     : raw logits from model.step() before bag, stored per frame
    post_bag_probs     : aggregated probability vector after bag, stored per frame
                         None for first (bag_size - 1) frames until bag is full
    emit_region        : (start_frame, end_frame, label) on emission, else None
    region_start_frame : frame index where current IN_SIGN region began
    """

    def __init__(
        self,
        id_to_label: dict[int, str],
        background_label: str,
        bag_size: int               = BAG_SIZE,
        aggregation: str            = BAG_AGGREGATION,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        sign_bg_margin: float       = SIGN_BG_MARGIN,
        min_sign_frames: int        = MIN_SIGN_FRAMES,
    ):
        self.id_to_label          = id_to_label
        self.background_label     = background_label
        self.confidence_threshold = float(confidence_threshold)
        self.sign_bg_margin       = float(sign_bg_margin)
        self.min_sign_frames      = max(1, int(min_sign_frames))

        self.background_id = next(
            (k for k, v in id_to_label.items() if v == background_label), None
        )

        self._bag = _BagAggregator(bag_size, aggregation, len(id_to_label))

        # Hysteresis state
        self.state              = "SEEKING"
        self.region_votes       = Counter()
        self.sign_frames        = 0
        self.region_start_frame = None      # frame where current IN_SIGN region began

    # ------------------------------------------------------------------

    def _gate(self, agg_probs: np.ndarray):
        """
        Apply confidence gate to aggregated probabilities.

        Returns
        -------
        voted_label, is_background, pred_label, pred_conf, bg_conf, agg_probs
        agg_probs passed through so caller can store it as post_bag_probs
        without recomputing.
        """
        pred_id    = int(np.argmax(agg_probs))
        pred_label = self.id_to_label.get(pred_id, f"sign_{pred_id}")
        pred_conf  = float(agg_probs[pred_id])
        bg_conf    = (
            float(agg_probs[self.background_id])
            if self.background_id is not None else 0.0
        )

        is_sign = (
            pred_label != self.background_label
            and pred_conf  >= self.confidence_threshold
            and (pred_conf - bg_conf) >= self.sign_bg_margin
        )

        voted_label   = pred_label if is_sign else self.background_label
        is_background = not is_sign

        return voted_label, is_background, pred_label, pred_conf, bg_conf, agg_probs

    # ------------------------------------------------------------------

    def update(self, logits: np.ndarray, frame_index: int) -> dict:
        """
        Process one frame.

        Parameters
        ----------
        logits      : (C,) raw logits from model.step() — stored as pre_bag_logits
        frame_index : int current frame index, needed for emit_region tracking

        Returns
        -------
        dict containing:
            raw_label      : top-1 label from pre-bag logits
            raw_conf       : top-1 confidence from pre-bag logits
            bg_conf        : background confidence from post-bag probs (0 until bag full)
            gated_label    : label after confidence gate (post-bag)
            voted_label    : same as gated_label
            state          : decoder state after this step (SEEKING / IN_SIGN)
            emitted_label  : emitted sign label if trailing edge fired, else None
            emit_region    : (start_frame, end_frame, label) on emission, else None
            pre_bag_logits : (C,) raw logits before bag — for visualization
            post_bag_probs : (C,) aggregated probs after bag — for visualization
                             None for first (bag_size - 1) frames
        """
        pre_bag_logits = logits.copy()              # capture before bag sees it
        agg_probs      = self._bag.update(logits)

        # Bag not full yet — stay in SEEKING, emit nothing
        if agg_probs is None:
            raw_probs = np.exp(logits - logits.max())
            raw_probs /= raw_probs.sum()
            return {
                "raw_label":      self.id_to_label.get(int(np.argmax(logits)), "?"),
                "raw_conf":       float(raw_probs.max()),
                "bg_conf":        0.0,
                "gated_label":    self.background_label,
                "voted_label":    self.background_label,
                "state":          self.state,
                "emitted_label":  None,
                "emit_region":    None,
                "pre_bag_logits": pre_bag_logits,   # (C,) always stored
                "post_bag_probs": None,             # bag not full yet
            }

        voted_label, is_background, pred_label, pred_conf, bg_conf, agg_probs = \
            self._gate(agg_probs)

        emitted_label = None
        emit_region   = None

        if self.state == "SEEKING":
            if not is_background:
                self.state              = "IN_SIGN"
                self.region_votes[voted_label] += 1
                self.sign_frames        = 1
                self.region_start_frame = frame_index   # record region start

        elif self.state == "IN_SIGN":
            if not is_background:
                self.region_votes[voted_label] += 1
                self.sign_frames += 1
            else:
                # Bag confirmed background — trailing edge reached
                if self.sign_frames >= self.min_sign_frames:
                    emitted_label = self.region_votes.most_common(1)[0][0]
                    emit_region   = (
                        self.region_start_frame,    # start of region
                        frame_index,                # end of region (trailing edge)
                        emitted_label,
                    )
                # else: region too short → discard silently

                self.state              = "SEEKING"
                self.region_votes       = Counter()
                self.sign_frames        = 0
                self.region_start_frame = None

        return {
            "raw_label":      pred_label,
            "raw_conf":       pred_conf,
            "bg_conf":        bg_conf,
            "gated_label":    voted_label,
            "voted_label":    voted_label,
            "state":          self.state,
            "emitted_label":  emitted_label,
            "emit_region":    emit_region,          # (start, end, label) or None
            "pre_bag_logits": pre_bag_logits,       # (C,) raw pre-bag logits
            "post_bag_probs": agg_probs,            # (C,) post-bag aggregated probs
        }

    # ------------------------------------------------------------------

    def flush(self) -> tuple[str | None, tuple | None]:
        """
        Call once after all frames are processed.

        Emits any sign region still open at sequence end.
        Necessary when a sequence ends without returning to background.

        Returns
        -------
        (emitted_label, emit_region)
        emit_region end frame is None — caller fills with t_len - 1.
        """
        emitted     = None
        emit_region = None

        if self.state == "IN_SIGN" and self.sign_frames >= self.min_sign_frames:
            emitted     = self.region_votes.most_common(1)[0][0]
            emit_region = (
                self.region_start_frame,
                None,       # end unknown — caller fills with t_len - 1
                emitted,
            )

        # Always reset — decoder is invalid after flush
        self.state              = "SEEKING"
        self.region_votes       = Counter()
        self.sign_frames        = 0
        self.region_start_frame = None
        self._bag.reset()

        return emitted, emit_region


# ---------------------------------------------------------------------------
# WER helpers — unchanged
# ---------------------------------------------------------------------------

def remove_consecutive_duplicates(labels: list[str]) -> list[str]:
    if not labels:
        return []
    out = [labels[0]]
    for lbl in labels[1:]:
        if lbl != out[-1]:
            out.append(lbl)
    return out


def remove_background(
    labels: list[str],
    background_label: str = BACKGROUND_LABEL,
) -> list[str]:
    return [lbl for lbl in labels if lbl != background_label]


def compute_wer(pred: list[str], gt: list[str]) -> float:
    n, m = len(gt), len(pred)
    if n == 0:
        return 0.0 if m == 0 else 1.0

    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    for i in range(1, n + 1):
        dp[i, 0] = i
    for j in range(1, m + 1):
        dp[0, j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost     = 0 if gt[i - 1] == pred[j - 1] else 1
            dp[i, j] = min(dp[i-1, j] + 1, dp[i, j-1] + 1, dp[i-1, j-1] + cost)

    return float(dp[n, m] / n)


# ---------------------------------------------------------------------------
# LSTM streaming inference
# ---------------------------------------------------------------------------

def stream_lstm_online(
    V: np.ndarray,
    model_obj: nn.Module,
    normalize_fn,
    id_to_label: dict[int, str],
    background_label: str,
    bag_size: int               = BAG_SIZE,
    aggregation: str            = BAG_AGGREGATION,
    confidence_threshold: float = CONFIDENCE_THRESHOLD,
    sign_bg_margin: float       = SIGN_BG_MARGIN,
    min_sign_frames: int        = MIN_SIGN_FRAMES,
) -> tuple[list[dict], list[str], list[tuple]]:
    """
    Frame-by-frame LSTM streaming inference with bag-aggregated decoder.

    Key difference from THCT-Net version: no sliding window or stride.
    model.step() processes one frame at a time carrying hidden state forward.
    logits from model.step() are the pre-bag logits stored in each step dict.

    Parameters
    ----------
    V            : (T, D) input feature array
    model_obj    : LSTM model with a .step(frame_t, hidden) method
    normalize_fn : per-frame normalization callable

    Returns
    -------
    stream_steps  : list of per-frame dicts, each containing:
                        raw_label, raw_conf, bg_conf,
                        gated_label, voted_label, state,
                        emitted_label, emit_region,
                        pre_bag_logits, post_bag_probs,
                        frame_index
    emitted_preds : ordered list of emitted sign labels
    emit_regions  : list of (start_frame, end_frame, label) — one per emission
    """
    if V.ndim != 2:
        raise ValueError(f"Expected (T, D), got {V.shape}")

    t_len = V.shape[0]
    if t_len == 0:
        return [], [], []

    decoder = SimplifiedBagDecoder(
        id_to_label=id_to_label,
        background_label=background_label,
        bag_size=bag_size,
        aggregation=aggregation,
        confidence_threshold=confidence_threshold,
        sign_bg_margin=sign_bg_margin,
        min_sign_frames=min_sign_frames,
    )

    model_obj.eval()
    hidden         = None
    stream_steps   : list[dict]  = []
    emitted_preds  : list[str]   = []
    emit_regions   : list[tuple] = []

    with torch.no_grad():
        for frame_idx in range(t_len):
            frame   = normalize_fn(
                V[frame_idx : frame_idx + 1].astype(np.float32, copy=False)
            )
            frame_t = torch.tensor(frame, dtype=torch.float32, device=DEVICE)

            # logits_np is the pre-bag raw logit from the LSTM
            logits, hidden = model_obj.step(frame_t, hidden)
            logits_np      = logits[0].detach().cpu().numpy().astype(np.float32)

            decoded                = decoder.update(logits_np, frame_index=frame_idx)
            decoded["frame_index"] = int(frame_idx)
            stream_steps.append(decoded)

            if decoded["emitted_label"] is not None:
                emitted_preds.append(decoded["emitted_label"])
            if decoded["emit_region"] is not None:
                emit_regions.append(decoded["emit_region"])

    # Flush — emit any sign still open at sequence end
    final_emission, final_emit_region = decoder.flush()

    if final_emission is not None:
        # Fill the None end frame from flush with the last frame index
        if final_emit_region is not None:
            final_emit_region = (
                final_emit_region[0],
                t_len - 1,
                final_emit_region[2],
            )
        emitted_preds.append(final_emission)
        emit_regions.append(final_emit_region)
        stream_steps.append({
            "raw_label":      final_emission,
            "raw_conf":       1.0,
            "bg_conf":        0.0,
            "gated_label":    final_emission,
            "voted_label":    final_emission,
            "state":          "FLUSH",
            "emitted_label":  final_emission,
            "emit_region":    final_emit_region,    # (start, t_len-1, label)
            "pre_bag_logits": None,                 # no new frame at flush
            "post_bag_probs": None,
            "frame_index":    t_len - 1,
        })

    return stream_steps, emitted_preds, emit_regions


# ---------------------------------------------------------------------------
# Summary printer
# ---------------------------------------------------------------------------

def _print_wer_summary(
    df: pd.DataFrame,
    split_name: str,
    normalization_name: str,
    stream_mode: str,
    print_examples: int,
) -> None:
    header = f"========== {split_name.upper()} WER SUMMARY =========="
    print(f"\n{header}")
    print(f"Normalization  : {normalization_name}")
    print(f"Stream mode    : {stream_mode}")
    print(f"Bag size       : {BAG_SIZE}  ({BAG_SIZE/LEAP_FPS*1000:.0f}ms)")
    print(f"Aggregation    : {BAG_AGGREGATION}")
    print(f"Conf threshold : {CONFIDENCE_THRESHOLD}")
    print(f"Min sign frames: {MIN_SIGN_FRAMES}  ({MIN_SIGN_MS}ms)")
    print(f"Total sequences evaluated: {len(df)}")

    if df.empty:
        return

    print(f"Mean WER:   {df['wer'].mean():.4f}")
    print(f"Median WER: {df['wer'].median():.4f}")
    print(f"Std WER:    {df['wer'].std(ddof=0):.4f}")

    n_show = min(print_examples, len(df))
    print(f"\nShowing {n_show} example sequence(s):")
    for _, row in df.head(n_show).iterrows():
        print(f"  [{int(row['sample_idx'])}] {row['user']} | {row['recording_id']}")
        print(f"    Frames       : {int(row['num_frames'])} | "
              f"Missing ratio: {float(row['missing_ratio']):.3f}")
        print(f"    Stream steps : {int(row['num_stream_predictions'])} | "
              f"Emitted: {int(row['emitted_count'])}")
        print(f"    GT           : {row['ground_truth'].split()}")
        print(f"    Prediction   : {row['prediction'].split()}")
        print(f"    WER          : {float(row['wer']):.4f}")
        if row.get("emit_regions"):
            print(f"    Emit regions : {row['emit_regions']}")
        if row.get("gt_segments"):
            print(f"    GT segments  : {row['gt_segments']}")


# ---------------------------------------------------------------------------
# Evaluation harness
# ---------------------------------------------------------------------------

def evaluate_lstm_wer(
    samples: list[dict],
    split_name: str,
    model_obj: nn.Module,
    normalize_fn: Callable[[np.ndarray], np.ndarray],
    normalization_name: str,
    print_examples: int         = 2,
    bag_size: int               = BAG_SIZE,
    aggregation: str            = BAG_AGGREGATION,
    confidence_threshold: float = CONFIDENCE_THRESHOLD,
    sign_bg_margin: float       = SIGN_BG_MARGIN,
    min_sign_frames: int        = MIN_SIGN_FRAMES,
) -> pd.DataFrame:
    """
    Evaluate streaming WER for an entire LSTM sequence set.

    Parameters
    ----------
    samples : list of dicts with keys:
                V, ground_truth, user, recording_id,
                num_frames, missing_ratio,
                segmentation_regions   ← used for GT visualization

    Returns
    -------
    pd.DataFrame — one row per sample, including:
        All original scalar WER fields
        stream_steps   : full per-frame decoder output (heavy — saved to npz)
        emit_regions   : [(start_frame, end_frame, label), ...] lightweight
        gt_segments    : from sample["segmentation_regions"]   lightweight
    """
    rows = []

    for idx, sample in enumerate(samples, start=1):
        stream_steps, final_preds, emit_regions = stream_lstm_online(
            V=sample["V"],
            model_obj=model_obj,
            normalize_fn=normalize_fn,
            id_to_label=id_to_label,
            background_label=BACKGROUND_LABEL,
            bag_size=bag_size,
            aggregation=aggregation,
            confidence_threshold=confidence_threshold,
            sign_bg_margin=sign_bg_margin,
            min_sign_frames=min_sign_frames,
        )

        wer         = compute_wer(final_preds, sample["ground_truth"])
        gt_segments = sample.get("segmentation_regions", [])

        rows.append({
            # ---- all original scalar fields ----
            "sample_idx":             idx,
            "split":                  split_name,
            "user":                   sample["user"],
            "recording_id":           sample["recording_id"],
            "num_frames":             sample["num_frames"],
            "missing_ratio":          sample["missing_ratio"],
            "gt_len":                 len(sample["ground_truth"]),
            "pred_len":               len(final_preds),
            "raw_len":                len(stream_steps),
            "wer":                    wer,
            "stream_mode":            STREAM_MODE,
            "stream_delay_frames":    0,
            "num_stream_predictions": len(stream_steps),
            "first_prediction_frame": (
                int(stream_steps[0]["frame_index"]) if stream_steps else None
            ),
            "emitted_count":          len(final_preds),
            "ground_truth":           " ".join(sample["ground_truth"]),
            "prediction":             " ".join(final_preds),
            # ---- visualization extras ----
            "stream_steps":           stream_steps,     # heavy — stripped before parquet
            "emit_regions":           emit_regions,     # lightweight list of tuples
            "gt_segments":            gt_segments,      # lightweight list from sample
        })

    df = pd.DataFrame(rows)
    _print_wer_summary(df, split_name, normalization_name, STREAM_MODE, print_examples)
    return df



# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if len(test_wer_catalog) == 0:
    raise RuntimeError(f"No WER sequences available for test user '{TEST_USER}'.")

print("========== ONLINE WER SETUP ==========")
print(f"Model                 : {MODEL_NAME}")
print(f"Normalization         : {NORMALIZATION_NAME}")
print(f"Test sequences        : {len(test_wer_catalog)} ({TEST_USER})")
print(f"Dev val sequences     : {len(dev_val_wer_catalog)}")
print(f"Dev train sequences   : {len(dev_train_wer_catalog)} (WER optional)")
print(f"Printed examples/split: {WER_EXAMPLE_PRINT_COUNT}")
print(f"Stream mode           : {STREAM_MODE}")
print(f"Bag size              : {BAG_SIZE} frames (~{BAG_SIZE/LEAP_FPS*1000:.0f}ms)")
print(f"Aggregation           : {BAG_AGGREGATION}")
print(f"Confidence threshold  : {CONFIDENCE_THRESHOLD}")
print(f"Min sign frames       : {MIN_SIGN_FRAMES} (~{MIN_SIGN_MS}ms)")

test_wer_df = evaluate_lstm_wer(
    samples=test_wer_catalog,
    split_name=f"Test ({TEST_USER})",
    model_obj=model,
    normalize_fn=NORMALIZE_FN,
    normalization_name=NORMALIZATION_NAME,
    print_examples=WER_EXAMPLE_PRINT_COUNT,
)

val_wer_df = evaluate_lstm_wer(
    samples=dev_val_wer_catalog,
    split_name="Dev val",
    model_obj=model,
    normalize_fn=NORMALIZE_FN,
    normalization_name=NORMALIZATION_NAME,
    print_examples=WER_EXAMPLE_PRINT_COUNT,
)

train_wer_df = pd.DataFrame()

# ------------------------------------------------------------------
# Save all splits to disk immediately after evaluation
# Heavy stream_steps arrays → npz; all scalars + lists → parquet
# ------------------------------------------------------------------


wer_comparison_df = pd.DataFrame([{
    "model_name":           MODEL_NAME,
    "normalization":        NORMALIZATION_NAME,
    "stream_mode":          STREAM_MODE,
    "bag_size":             BAG_SIZE,
    "aggregation":          BAG_AGGREGATION,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "min_sign_frames":      MIN_SIGN_FRAMES,
    "test_num_sequences":   int(len(test_wer_df)),
    "test_mean_wer":        float(test_wer_df["wer"].mean())   if len(test_wer_df) else float("nan"),
    "test_median_wer":      float(test_wer_df["wer"].median()) if len(test_wer_df) else float("nan"),
    "val_num_sequences":    int(len(val_wer_df)),
    "val_mean_wer":         float(val_wer_df["wer"].mean())    if len(val_wer_df)  else float("nan"),
    "val_median_wer":       float(val_wer_df["wer"].median())  if len(val_wer_df)  else float("nan"),
    "train_num_sequences":  int(len(train_wer_df)),
    "train_mean_wer":       float("nan"),
    "train_median_wer":     float("nan"),
}])

print("\n========== WER SUMMARY TABLE (LOWER IS BETTER) ==========")
display(wer_comparison_df)

online_eval_df   = test_wer_df.copy()
online_eval_rows = online_eval_df.to_dict(orient="records")

In [ ]:
RESULTS_DIR = "./wer_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def save_split_results(
    df: pd.DataFrame,
    split_name: str,
    results_dir: str = RESULTS_DIR,
) -> dict[str, str]:
    """
    Save all DataFrame columns + per-frame logit arrays for one split.

    Two files per split:
        {split}_metadata.parquet  — all scalar/list columns, one row per sequence
        {split}_arrays.npz        — per-frame arrays stacked across all sequences
                                    indexed by recording_id

    Parameters
    ----------
    df           : DataFrame returned by evaluate_sequence_set_wer()
    split_name   : "test" | "val" | "train" — used as filename prefix
    results_dir  : directory to write into

    Returns
    -------
    dict with paths to the two saved files
    """
    split_slug = split_name.lower().replace(" ", "_").replace("(", "").replace(")", "")

    # ------------------------------------------------------------------
    # 1. Separate heavy per-frame arrays from the rest
    # ------------------------------------------------------------------
    df_meta = df.drop(columns=["stream_steps"], errors="ignore").copy()

    # emit_regions and gt_segments are lists of tuples — not parquet-native
    # serialize them to JSON strings so parquet can store them cleanly
    for col in ["emit_regions", "gt_segments"]:
        if col in df_meta.columns:
            df_meta[col] = df_meta[col].apply(
                lambda x: json.dumps(x) if x is not None else "[]"
            )

    # ------------------------------------------------------------------
    # 2. Save metadata DataFrame — all WER scalars, GT, predictions, etc.
    # ------------------------------------------------------------------
    parquet_path = os.path.join(results_dir, f"{split_slug}_metadata.parquet")
    df_meta.to_parquet(parquet_path, index=False)

    # ------------------------------------------------------------------
    # 3. Save per-frame arrays — one block per sequence, stored by index
    #    so they can be looked up by row position in the DataFrame
    # ------------------------------------------------------------------
    arrays_dict = {}

    for row_idx, row in df.iterrows():
        steps = row.get("stream_steps")
        if not steps:
            continue

        rec_id = str(row["recording_id"]).replace("/", "_")  # safe key
        prefix = f"{row_idx}__{rec_id}"                       # unique even if rec_id repeats

        C = None  # num classes — inferred from first non-None logit

        pre_bag_list  = []
        post_bag_list = []

        for s in steps:
            pre  = s.get("pre_bag_logits")
            post = s.get("post_bag_probs")

            if C is None:
                if pre  is not None: C = len(pre)
                elif post is not None: C = len(post)

            pre_bag_list.append(
                pre if pre is not None
                else np.full(C or 1, np.nan, dtype=np.float32)
            )
            post_bag_list.append(
                post if post is not None
                else np.full(C or 1, np.nan, dtype=np.float32)
            )

        pre_bag  = np.stack(pre_bag_list,  axis=0).astype(np.float32)   # (T, C)
        post_bag = np.stack(post_bag_list, axis=0).astype(np.float32)   # (T, C)

        arrays_dict[f"{prefix}__pre_bag_logits"]  = pre_bag
        arrays_dict[f"{prefix}__post_bag_probs"]  = post_bag
        arrays_dict[f"{prefix}__frame_indices"]   = np.array(
            [s["frame_index"]  for s in steps], dtype=np.int32
        )
        arrays_dict[f"{prefix}__window_starts"]   = np.array(
            [s["window_start"] for s in steps], dtype=np.int32
        )
        arrays_dict[f"{prefix}__window_ends"]     = np.array(
            [s["window_end"]   for s in steps], dtype=np.int32
        )
        # lightweight per-frame label/conf — cheap to store alongside arrays
        arrays_dict[f"{prefix}__raw_labels"]      = np.array(
            [s["raw_label"]    for s in steps], dtype=object
        )
        arrays_dict[f"{prefix}__voted_labels"]    = np.array(
            [s["voted_label"]  for s in steps], dtype=object
        )
        arrays_dict[f"{prefix}__raw_conf"]        = np.array(
            [float(s["raw_conf"])  for s in steps], dtype=np.float32
        )
        arrays_dict[f"{prefix}__bg_conf"]         = np.array(
            [float(s["bg_conf"])   for s in steps], dtype=np.float32
        )
        arrays_dict[f"{prefix}__states"]          = np.array(
            [s["state"]        for s in steps], dtype=object
        )

    npz_path = os.path.join(results_dir, f"{split_slug}_arrays.npz")
    np.savez_compressed(npz_path, **arrays_dict)

    print(f"[{split_name}] metadata → {parquet_path}")
    print(f"[{split_name}] arrays   → {npz_path}")
    print(f"  sequences : {len(df)}")
    print(f"  npz keys  : {len(arrays_dict)}")

    return {"parquet": parquet_path, "npz": npz_path}


def load_split_results(
    split_name: str,
    results_dir: str = RESULTS_DIR,
) -> tuple[pd.DataFrame, dict]:
    """
    Load metadata DataFrame and per-frame arrays for one split.

    Returns
    -------
    df    : DataFrame with all scalar/list columns restored
            emit_regions and gt_segments are deserialized back to lists
    arrays: dict — keys are "{row_idx}__{rec_id}__{field}", values are np.ndarray
            Access example:
                arrays["0__rec001__pre_bag_logits"]  → (T, C) float32
                arrays["0__rec001__frame_indices"]   → (T,)   int32
    """
    split_slug   = split_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    parquet_path = os.path.join(results_dir, f"{split_slug}_metadata.parquet")
    npz_path     = os.path.join(results_dir, f"{split_slug}_arrays.npz")

    df = pd.read_parquet(parquet_path)

    # Deserialize JSON strings back to Python lists
    for col in ["emit_regions", "gt_segments"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: [tuple(r) for r in json.loads(x)] if isinstance(x, str) else []
            )

    npz  = np.load(npz_path, allow_pickle=True)
    arrays = {k: npz[k] for k in npz.files}

    return df, arrays


def get_sequence_arrays(
    row: pd.Series,
    arrays: dict,
) -> dict:
    """
    Convenience helper — extract all per-frame arrays for one DataFrame row.

    Usage
    -----
    df, arrays = load_split_results("test")
    seq = get_sequence_arrays(df.iloc[0], arrays)
    seq["pre_bag_logits"]   # (T, C)
    seq["post_bag_probs"]   # (T, C)  first (bag_size-1) rows are NaN
    seq["frame_indices"]    # (T,)
    """
    row_idx = row.name  # DataFrame index
    rec_id  = str(row["recording_id"]).replace("/", "_")
    prefix  = f"{row_idx}__{rec_id}"

    fields = [
        "pre_bag_logits", "post_bag_probs",
        "frame_indices",  "window_starts", "window_ends",
        "raw_labels",     "voted_labels",
        "raw_conf",       "bg_conf",
        "states",
    ]
    return {
        field: arrays.get(f"{prefix}__{field}")
        for field in fields
    }

In [ ]:
# Save all three splits
test_paths  = save_split_results(test_wer_df,  split_name="test")
val_paths   = save_split_results(val_wer_df,   split_name="val")
# train_wer_df is empty but save anyway so the pattern is consistent
if not train_wer_df.empty:
    train_paths = save_split_results(train_wer_df, split_name="train")


# test_df, test_arrays = load_split_results("test")

# # one sequence
# row = test_df.iloc[0]
# seq = get_sequence_arrays(row, test_arrays)

# print(row["wer"])                    # 0.25
# print(row["ground_truth"])           # "HELLO WORLD"
# print(row["emit_regions"])           # [(12, 89, "HELLO"), (120, 210, "WORLD")]
# print(row["gt_segments"])            # from sample["segmentation_regions"]
# print(seq["pre_bag_logits"].shape)   # (270, 100)
# print(seq["post_bag_probs"].shape)   # (270, 100) — first 4 rows NaN

## 18) Save Model


In [ ]:
def save_unique_model(
    model_obj: nn.Module,
    save_dir: str = "trained_models",
    model_name: str = MODEL_NAME,
    info: dict | None = None,
) -> str:
    os.makedirs(save_dir, exist_ok=True)
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    short_id = uuid.uuid4().hex[:8]
    best_val = None
    try:
        best_val = (
            float(train_result.get("best_val_acc"))
            if "train_result" in globals() and train_result.get("best_val_acc") is not None
            else None
        )
    except Exception:
        best_val = None

    val_str = f"{best_val:.4f}" if best_val is not None else "nan"
    filename = f"{ts}_{model_name}_val-{val_str}_{short_id}.pt"
    path = os.path.join(save_dir, filename)

    metadata = {
        "model_name": model_name,
        "saved_at_utc": ts,
        "uid": short_id,
        "best_val_acc": best_val,
        "normalization": NORMALIZATION_NAME,
        "notebook": notebook_name if "notebook_name" in globals() else None,
    }

    try:
        git_sha = subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=str(PROJECT_ROOT),
            stderr=subprocess.DEVNULL,
        ).decode().strip()
        metadata["git_sha"] = git_sha
    except Exception:
        pass

    if info:
        metadata.update(info)

    payload = {"model_state_dict": model_obj.state_dict(), "metadata": metadata}
    torch.save(payload, path)

    try:
        with open(path + ".json", "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)
    except Exception:
        pass

    print(f"Saved model checkpoint: {path}")
    return path


if "model" in globals():
    test_mean_wer = float(test_wer_df["wer"].mean()) if "test_wer_df" in globals() and len(test_wer_df) else None
    saved_path = save_unique_model(
        model,
        save_dir="trained_models",
        model_name=MODEL_NAME,
        info={
            "source": "prototype_sequence_lstm_palm_ref_users3.ipynb",
            "feat_dim": FEAT_DIM,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LSTM_LAYERS,
            "test_mean_wer": test_mean_wer,
        },
    )
else:
    print("No model object found in globals(); skipping save.")
